# ALDIMI - Merge salud y stock

Este notebook prepara el dataset de stock con la estructura requerida y lo vincula con el dataset de salud.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
BASE_DIR = Path.cwd()
if BASE_DIR.name == 'src':
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
MERGED_DIR = DATA_DIR / 'merged'

for d in (RAW_DIR, PROCESSED_DIR, MERGED_DIR):
    d.mkdir(parents=True, exist_ok=True)

health_df = pd.read_csv(RAW_DIR / 'health_raw.csv')
stock_raw = pd.read_csv(RAW_DIR / 'stock_raw.csv')

print('Health shape:', health_df.shape)
print('Stock shape:', stock_raw.shape)

Health shape: (2000, 21)
Stock shape: (91250, 15)


In [3]:
def find_col(df, keywords):
    for col in df.columns:
        low = col.lower()
        if any(k in low for k in keywords):
            return col
    return None

def ensure_fecha(df):
    df = df.copy()
    date_col = find_col(df, ['date', 'fecha', 'day'])
    if date_col:
        df['Fecha'] = pd.to_datetime(df[date_col], errors='coerce')
        df['Fecha'] = df['Fecha'].fillna(pd.Timestamp('2023-01-01'))
    else:
        df['Fecha'] = pd.date_range('2023-01-01', periods=len(df), freq='D')
    return df

In [4]:
# Salud: crear Alto_Riesgo y ocupacion total por fecha
health_df = ensure_fecha(health_df)
risk_col = find_col(health_df, ['risk_level', 'risk level', 'risklevel'])
score_col = find_col(health_df, ['risk_score', 'overall_risk', 'score'])

if risk_col:
    health_df['Alto_Riesgo'] = health_df[risk_col].astype(str).str.lower().str.contains('high')
elif score_col:
    threshold = health_df[score_col].median()
    health_df['Alto_Riesgo'] = pd.to_numeric(health_df[score_col], errors='coerce') >= threshold
else:
    health_df['Alto_Riesgo'] = False

health_daily = health_df.groupby('Fecha').agg(
    Pacientes_Alto_Riesgo=('Alto_Riesgo', 'sum'),
    Ocupacion_Total=('Alto_Riesgo', 'count')
).reset_index()

health_daily.to_csv(PROCESSED_DIR / 'health_daily.csv', index=False)
print('Saved health daily:', PROCESSED_DIR / 'health_daily.csv')
health_daily.head()

Saved health daily: d:\Gitproyectos\Machine-learning\data\processed\health_daily.csv


,Fecha,Pacientes_Alto_Riesgo,Ocupacion_Total
0,2023-01-01,0,1
1,2023-01-02,0,1
2,2023-01-03,0,1
3,2023-01-04,0,1
4,2023-01-05,0,1


In [5]:
# Stock: estructurar columnas requeridas
stock_df = ensure_fecha(stock_raw)
item_col = find_col(stock_df, ['item', 'sku', 'product', 'category', 'material', 'name'])
stock_level_col = find_col(stock_df, ['stock', 'inventory', 'onhand', 'on_hand', 'quantity', 'qty'])
lead_col = find_col(stock_df, ['lead', 'replenish', 'reorder', 'lt'])

if item_col:
    item_str = stock_df[item_col].astype(str).str.lower()
    stock_df['ID_Insumo'] = np.where(
        item_str.str.contains('drug|med|pharma|medicine'),
        'medicinas',
        'alimentos'
    )
else:
    stock_df['ID_Insumo'] = 'alimentos'

if stock_level_col:
    stock_df['Stock_Actual'] = pd.to_numeric(stock_df[stock_level_col], errors='coerce').fillna(0)
else:
    rng = np.random.default_rng(42)
    stock_df['Stock_Actual'] = rng.integers(50, 500, size=len(stock_df))

if lead_col:
    stock_df['Lead_Time'] = pd.to_numeric(stock_df[lead_col], errors='coerce').fillna(7).astype(int)
else:
    rng = np.random.default_rng(42)
    stock_df['Lead_Time'] = rng.integers(2, 15, size=len(stock_df))

stock_base = stock_df[['Fecha', 'ID_Insumo', 'Stock_Actual', 'Lead_Time']].copy()
stock_with_health = stock_base.merge(health_daily, on='Fecha', how='left')

stock_with_health['Pacientes_Alto_Riesgo'] = stock_with_health['Pacientes_Alto_Riesgo'].fillna(0)
stock_with_health['Ocupacion_Total'] = stock_with_health['Ocupacion_Total'].fillna(0)

rng = np.random.default_rng(42)
noise = rng.normal(0, 5, size=len(stock_with_health))
stock_with_health['Consumo_Diario'] = (
    20
    + 0.4 * stock_with_health['Pacientes_Alto_Riesgo']
    + 0.1 * stock_with_health['Ocupacion_Total']
    + noise
).clip(lower=1).round(2)

stock_with_health['Ocupacion_Albergue'] = (
    0.6 * stock_with_health['Ocupacion_Total']
    + rng.normal(0, 2, size=len(stock_with_health))
).clip(lower=0).round(2)

stock_structured = stock_with_health[[
    'Fecha',
    'ID_Insumo',
    'Stock_Actual',
    'Consumo_Diario',
    'Lead_Time',
    'Ocupacion_Albergue'
]].copy()

stock_structured.to_csv(PROCESSED_DIR / 'stock_structured.csv', index=False)
print('Saved stock structured:', PROCESSED_DIR / 'stock_structured.csv')
stock_structured.head()

Saved stock structured: d:\Gitproyectos\Machine-learning\data\processed\stock_structured.csv


,Fecha,ID_Insumo,Stock_Actual,Consumo_Diario,Lead_Time,Ocupacion_Albergue
0,2024-01-01,alimentos,592,21.62,14,0.68
1,2024-01-02,alimentos,575,14.90,14,0.00
2,2024-01-03,alimentos,540,23.85,14,0.00
3,2024-01-04,alimentos,516,25.20,14,1.69
4,2024-01-05,alimentos,495,10.34,14,0.00


In [6]:
# Merge final de salud y stock
merged = stock_structured.merge(health_daily, on='Fecha', how='left')
merged_out = MERGED_DIR / 'Dataset_ALDIMI_Merged.csv'
merged.to_csv(merged_out, index=False)
print('Saved merged dataset:', merged_out)

corr = merged[['Consumo_Diario', 'Pacientes_Alto_Riesgo', 'Ocupacion_Total']].corr()
print('Correlation check:')
print(corr)

merged.head()

Saved merged dataset: d:\Gitproyectos\Machine-learning\data\merged\Dataset_ALDIMI_Merged.csv
Correlation check:
                       Consumo_Diario  Pacientes_Alto_Riesgo  Ocupacion_Total
Consumo_Diario               1.000000               0.011057              NaN
Pacientes_Alto_Riesgo        0.011057               1.000000              NaN
Ocupacion_Total                   NaN                    NaN              NaN


,Fecha,ID_Insumo,Stock_Actual,Consumo_Diario,Lead_Time,Ocupacion_Albergue,Pacientes_Alto_Riesgo,Ocupacion_Total
0,2024-01-01,alimentos,592,21.62,14,0.68,0,1
1,2024-01-02,alimentos,575,14.90,14,0.00,0,1
2,2024-01-03,alimentos,540,23.85,14,0.00,0,1
3,2024-01-04,alimentos,516,25.20,14,1.69,1,1
4,2024-01-05,alimentos,495,10.34,14,0.00,0,1


Se procederá la enriquecimiento del dataset de logistica 

- "Dataset_ALDIMI_Merged.csv" es el dataset destinado a la predicción de si es urgente reabastecer el stock o no ("Necesita_Reabastecimiento"), en base al consumo del paciente y el stock que se presenta en ese momento

In [12]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(r"D:\Gitproyectos\Machine-learning\data\merged\Dataset_ALDIMI_Merged.csv")

print("Forma:", df.shape)

df["Punto_Reorden"] = df["Consumo_Diario"] * df["Lead_Time"]
df["Ratio_Stock"] = df["Stock_Actual"] / df["Punto_Reorden"]

def clasificar(fila):
    ratio = fila["Ratio_Stock"]

    if ratio > 5.6:
        return 0
    elif ratio > 2.2:
        return 1
    else:
        return 2

df["Necesita_Reabastecimiento"] = df.apply(clasificar, axis=1)

print("\nDistribución:")
print(df["Necesita_Reabastecimiento"].value_counts())

print("\nPorcentajes:")
print((df["Necesita_Reabastecimiento"].value_counts(normalize=True) * 100).round(2))

base_dir = Path.cwd()

if base_dir.name == "src":
    base_dir = base_dir.parent

output_path = base_dir / "data" / "processed"
output_path.mkdir(parents=True, exist_ok=True)

output_file = output_path / "Dataset_ALDIMI_Logistica_Enriquecido.csv"
df.to_csv(output_file, index=False)

print("\nDataset guardado en:")
print(output_file)


Forma: (91250, 8)

Distribución:
Necesita_Reabastecimiento
1    40750
2    27921
0    22579
Name: count, dtype: int64

Porcentajes:
Necesita_Reabastecimiento
1    44.66
2    30.60
0    24.74
Name: proportion, dtype: float64

Dataset guardado en:
d:\Gitproyectos\Machine-learning\data\processed\Dataset_ALDIMI_Logistica_Enriquecido.csv


Se creó la columna Punto de Reorden a partir del Consumo Diario y el Lead Time porque permite estimar el nivel mínimo de inventario necesario antes de que ocurra un desabastecimiento, lo cual es un criterio estándar en gestión logística. A partir de esta variable se construyó el Ratio de Stock, que compara el inventario actual con el nivel crítico requerido, permitiendo medir de forma relativa la “holgura” o “riesgo” de stock en cada registro.

La variable objetivo Necesita_Reabastecimiento se definió en tres niveles utilizando umbrales sobre el Ratio de Stock para representar estados operativos interpretables: suficiente, alerta y crítico. Los valores utilizados (2.2 y 5.6) se eligieron a partir de los percentiles del propio dataset, lo que permite adaptar la clasificación a la distribución real de los datos en lugar de imponer reglas arbitrarias. Esto ayuda a reducir el desbalance extremo de clases y mejora la capacidad del modelo (como Random Forest o XGBoost) para aprender patrones útiles sin sesgarse hacia la clase dominante.

Se procederá la enriquecimiento del dataset de gravedad de pacientes

- "Dataset_ALDIMI_Merged_Clean.csv" es el dataset destinado a la predicción de la graveda 

D:\Gitproyectos\Machine-learning\data\processed\Dataset_ALDIMI_Merged_Clean.csv

In [13]:
import pandas as pd

df = pd.read_csv("D:\Gitproyectos\Machine-learning\data\processed\Dataset_ALDIMI_Merged_Clean.csv")

print("TIPOS DE VARIABLES:")
print(df.dtypes)

TIPOS DE VARIABLES:
Patient_ID                   int64
Cancer_Type                 object
Age                          int64
Gender                       int64
Smoking                      int64
Alcohol_Use                  int64
Obesity                      int64
Family_History               int64
Diet_Red_Meat                int64
Diet_Salted_Processed        int64
Fruit_Veg_Intake             int64
Physical_Activity            int64
Air_Pollution                int64
Occupational_Hazards         int64
BRCA_Mutation                int64
H_Pylori_Infection           int64
Calcium_Intake               int64
Overall_Risk_Score         float64
BMI                        float64
Physical_Activity_Level      int64
Risk_Level                  object
county_STATE                 int64
county_CTYNAME              object
county_POPESTIMATE2015     float64
dtype: object


<>:3: SyntaxWarning: invalid escape sequence '\G'
<>:3: SyntaxWarning: invalid escape sequence '\G'
C:\Users\mcabr\AppData\Local\Temp\ipykernel_9784\2096391599.py:3: SyntaxWarning: invalid escape sequence '\G'
  df = pd.read_csv("D:\Gitproyectos\Machine-learning\data\processed\Dataset_ALDIMI_Merged_Clean.csv")


In [16]:
from pathlib import Path


df["Habitos_Riesgo"] = (
    df["Smoking"] +
    df["Alcohol_Use"] +
    df["Obesity"] +
    df["Air_Pollution"] +
    df["Occupational_Hazards"]
)

df["Riesgo_Clinico"] = (
    df["Family_History"] +
    df["BRCA_Mutation"] +
    df["H_Pylori_Infection"]
)

df["Factor_Protector"] = (
    df["Fruit_Veg_Intake"] +
    df["Physical_Activity"] +
    df["Calcium_Intake"]
)

df["Balance_Riesgo"] = (
    df["Habitos_Riesgo"] +
    df["Riesgo_Clinico"] -
    df["Factor_Protector"]
)

df["Edad_Rango"] = pd.cut(
    df["Age"],
    bins=[0, 30, 45, 60, 120],
    labels=["Joven", "Adulto", "Mayor", "Adulto_Mayor"]
)

print("Nuevas columnas creadas:")
print(df[[
    "Habitos_Riesgo",
    "Riesgo_Clinico",
    "Factor_Protector",
    "Balance_Riesgo",
    "Edad_Rango"
]].head())

base_dir = Path.cwd()

if base_dir.name == "src":
    base_dir = base_dir.parent

output_path = base_dir / "data" / "processed"
output_path.mkdir(parents=True, exist_ok=True)

output_file = output_path / "Dataset_ALDIMI_GravedadPaciente_Enriquecido.csv"
df.to_csv(output_file, index=False)

print("Dataset guardado en:")
print(output_file)

Nuevas columnas creadas:
   Habitos_Riesgo  Riesgo_Clinico  Factor_Protector  Balance_Riesgo  \
0              41               0                18              23   
1              22               0                15               7   
2              13               0                13               0   
3              25               0                18               7   
4              30               1                19              12   

     Edad_Rango  
0         Mayor  
1  Adulto_Mayor  
2  Adulto_Mayor  
3  Adulto_Mayor  
4         Mayor  
Dataset guardado en:
d:\Gitproyectos\Machine-learning\data\processed\Dataset_ALDIMI_GravedadPaciente_Enriquecido.csv


In [19]:
# CANTIDAD POR CLASE
print("Cantidad de registros por clase:\n")
print(df["Risk_Level"].value_counts())

# PORCENTAJE POR CLASE
print("\nPorcentaje de registros por clase:\n")
print(df["Risk_Level"].value_counts(normalize=True) * 100)

# TABLA RESUMEN (BONITO PARA INFORME)
resumen = pd.DataFrame({
    "Cantidad": df["Risk_Level"].value_counts(),
    "Porcentaje": df["Risk_Level"].value_counts(normalize=True) * 100
})

print("\nResumen completo:\n")
print(resumen)

Cantidad de registros por clase:

Risk_Level
Medium    2368
Low        484
High       148
Name: count, dtype: int64

Porcentaje de registros por clase:

Risk_Level
Medium    78.933333
Low       16.133333
High       4.933333
Name: proportion, dtype: float64

Resumen completo:

            Cantidad  Porcentaje
Risk_Level                      
Medium          2368   78.933333
Low              484   16.133333
High             148    4.933333


Se realizó un proceso de enriquecimiento y transformación de datos con el objetivo de mejorar la capacidad predictiva del modelo orientado a la clasificación del nivel de prioridad de atención (Risk_Level: Bajo, Medio y Alto). Este proceso consistió en la creación de nuevas variables derivadas a partir de las características originales del dataset, agrupando factores relacionados como hábitos de riesgo, condiciones clínicas y factores protectores. Estas nuevas variables permiten resumir información dispersa en indicadores compuestos más representativos del estado general del paciente, facilitando que el modelo identifique patrones no lineales de mayor complejidad.

Este enriquecimiento es clave porque mejora la calidad de la señal de los datos, reduce la fragmentación de variables individuales y permite capturar interacciones entre factores que por separado no serían evidentes. Como resultado, el modelo puede diferenciar con mayor precisión entre pacientes de bajo, medio y alto riesgo, especialmente en casos críticos donde múltiples factores clínicos y de estilo de vida influyen simultáneamente en la clasificación final.